# 43 LightGBM ウォークフォワード評価 (暗号資産)

| 項目 | 内容 |
|------|------|
| 入力 | `data/crypto_ohlcv.parquet`, `data/macro.parquet` |
| 対象 | BTC-USD, ETH-USD, SOL-USD, BNB-USD, XRP-USD |
| 予測ターゲット | `fwd_ret_7d` (7 日先のログリターン) |
| 評価指標 | Rank IC (スピアマン相関) |
| 出力 | `results/tables/crypto_lgbm_models.pkl` など |

## 株式 (42_) との主な違い
| 項目 | 株式 (42_) | 暗号資産 (本ノート) |
|------|-----------|--------------------|
| 市場時間 | 営業日のみ | 365日 24/7 |
| ターゲット | fwd_ret_5d | fwd_ret_1d / fwd_ret_7d |
| BTC先行効果 | なし | BTC_ret を altcoin の特徴量に追加 |
| 曜日効果 | 少ない | 週末ダンピング等あり → dow dummies |
| ボラティリティ | 年率20-40% | 年率60-150% |

## 使用特徴量
| カテゴリ | 特徴量 |
|----------|--------|
| リターン | ret_1d/3d/7d/14d/30d, ログリターン |
| 移動平均 | SMA 7/14/30/60, EMA 12/26, 価格比 |
| モメンタム | RSI 14, MACD (12/26/9) |
| ボラティリティ | vol_7d/14d/30d (年率換算), BB%B, BBW |
| ボリューム | vol_chg, vol_sma_ratio_7d, volume_zscore_14d |
| BTC先行 | BTC_ret_1d, BTC_ret_3d (非BTC銘柄のみ) |
| 曜日 | dow_0..6 ダミー |
| クロスセクション | 日次ランク (相対強度) |
| マクロ | USD index, gold, USDJPY のリターン |

---
## 0. 環境セットアップ

In [ ]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import yaml
from scipy.stats import spearmanr
from sklearn.model_selection import TimeSeriesSplit

try:
    import japanize_matplotlib
except ImportError:
    pass

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

print('Setup OK')

---
## 1. 設定・データ読み込み

In [ ]:
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR    = Path(cfg['paths']['data'])
FIGURES_DIR = Path(cfg['paths']['figures'])
TABLES_DIR  = Path(cfg['paths']['tables'])
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = cfg.get('random_seed', 42)
np.random.seed(RANDOM_SEED)

# 暗号資産ターゲット: 7日先ログリターン
TARGET_1D = 'fwd_ret_1d'
TARGET_7D = 'fwd_ret_7d'
TARGET    = TARGET_7D  # メインターゲット

N_SPLITS  = cfg['model']['n_splits']   # 5

# 暗号資産向けLGBMパラメータ（株式より滑らかに）
LGB_PARAMS = {
    'objective':         'regression',
    'n_estimators':      300,
    'max_depth':         5,
    'num_leaves':        31,   # 株式の20より多め（暗号資産のパターンはリッチ）
    'learning_rate':     0.03,  # 過学習抑制のため株式(0.05)より低く
    'colsample_bytree':  0.7,
    'subsample':         0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'min_child_samples': 10,   # 暗号資産は銘柄数が少ないため緩和
    'random_state':      RANDOM_SEED,
    'n_jobs':            -1,
    'verbose':           -1,
}

CRYPTO_SYMBOLS = cfg['crypto']['symbols']  # ['BTC-USD','ETH-USD','SOL-USD','BNB-USD','XRP-USD']

print(f'Target        : {TARGET}')
print(f'Crypto symbols: {CRYPTO_SYMBOLS}')
print(f'Splits        : {N_SPLITS}')
print(f'Tables        : {TABLES_DIR.resolve()}')

In [ ]:
df_raw   = pd.read_parquet(DATA_DIR / 'crypto_ohlcv.parquet')
df_macro = pd.read_parquet(DATA_DIR / 'macro.parquet')

# 日付型の確認・統一
df_raw['date']   = pd.to_datetime(df_raw['date'])
df_macro['date'] = pd.to_datetime(df_macro['date'])

# 対象銘柄のみ絞り込み
df_raw = df_raw[df_raw['symbol'].isin(CRYPTO_SYMBOLS)].copy()

print(f'OHLCV  : {len(df_raw):,} 行  {df_raw["symbol"].nunique()} 銘柄')
print(f'Macro  : {len(df_macro):,} 行  {df_macro.shape[1]-1} 指標')
print(f'OHLCV 期間: {df_raw["date"].min().date()} ~ {df_raw["date"].max().date()}')
print(f'銘柄: {sorted(df_raw["symbol"].unique().tolist())}')
df_raw.head(3)

---
## 2. 暗号資産向け特徴量エンジニアリング

暗号資産特有の要素:
- **24/7 市場**: 曜日ダミーで週末効果を捉える
- **高ボラティリティ**: 7/14/30d ウィンドウで短期のシグナルを優先
- **BTC先行効果**: BTC のリターンを altcoin の特徴量に追加
- **強いモメンタム & 平均回帰**: 複数期間のリターンを使う

In [ ]:
# =====================================================
# 特徴量計算関数
# =====================================================

def add_returns(df, windows=(1, 3, 7, 14, 30)):
    """ログリターン (複数ウィンドウ)"""
    for w in windows:
        df[f'ret_{w}d']    = df.groupby('symbol')['close'].pct_change(w)
        df[f'logret_{w}d'] = np.log(
            df.groupby('symbol')['close'].transform(lambda x: x / x.shift(w))
        )
    return df


def add_forward_returns(df, windows=(1, 7)):
    """フォワードリターン (ログ): ターゲット変数"""
    for w in windows:
        df[f'fwd_ret_{w}d'] = df.groupby('symbol')['close'].transform(
            lambda x: np.log(x.shift(-w) / x)
        )
    return df


def add_sma(df, windows=(7, 14, 30, 60)):
    """SMA / EMA + 価格比"""
    for w in windows:
        df[f'sma_{w}'] = df.groupby('symbol')['close'].transform(
            lambda x: x.rolling(w, min_periods=1).mean()
        )
    for span in (12, 26):
        df[f'ema_{span}'] = df.groupby('symbol')['close'].transform(
            lambda x: x.ewm(span=span, adjust=False).mean()
        )
    for w in windows:
        df[f'price_sma{w}_ratio'] = df['close'] / df[f'sma_{w}']
    return df


def add_rsi(df, window=14):
    """RSI (14日)"""
    delta = df.groupby('symbol')['close'].transform(lambda x: x.diff())
    gain  = delta.where(delta > 0, 0.0)
    loss  = -delta.where(delta < 0, 0.0)
    ag    = gain.groupby(df['symbol']).transform(
        lambda x: x.ewm(com=window - 1, min_periods=window).mean()
    )
    al    = loss.groupby(df['symbol']).transform(
        lambda x: x.ewm(com=window - 1, min_periods=window).mean()
    )
    rs = ag / al.replace(0, np.nan)
    df[f'rsi_{window}'] = 100 - (100 / (1 + rs))
    return df


def add_macd(df, fast=12, slow=26, signal=9):
    """MACD"""
    def _calc(x):
        ef = x.ewm(span=fast, adjust=False).mean()
        es = x.ewm(span=slow, adjust=False).mean()
        m  = ef - es
        s  = m.ewm(span=signal, adjust=False).mean()
        return pd.DataFrame({'macd': m, 'macd_signal': s, 'macd_hist': m - s})
    result = df.groupby('symbol')['close'].apply(_calc).reset_index(level=0, drop=True)
    return pd.concat([df, result], axis=1)


def add_bollinger(df, window=20, k=2.0):
    """ボリンジャーバンド (%B, 幅)"""
    df['bb_mid']   = df.groupby('symbol')['close'].transform(
        lambda x: x.rolling(window, min_periods=window // 2).mean()
    )
    df['bb_std']   = df.groupby('symbol')['close'].transform(
        lambda x: x.rolling(window, min_periods=window // 2).std()
    )
    df['bb_upper'] = df['bb_mid'] + k * df['bb_std']
    df['bb_lower'] = df['bb_mid'] - k * df['bb_std']
    band = (df['bb_upper'] - df['bb_lower']).replace(0, np.nan)
    df['bb_pct']   = (df['close'] - df['bb_lower']) / band
    df['bb_width'] = band / df['bb_mid']
    return df


def add_volatility(df, windows=(7, 14, 30)):
    """ヒストリカルボラティリティ (年率換算, 365日)"""
    lr = np.log(
        df.groupby('symbol')['close'].transform(lambda x: x / x.shift(1))
    )
    for w in windows:
        df[f'vol_{w}d'] = lr.groupby(df['symbol']).transform(
            lambda x: x.rolling(w, min_periods=w // 2).std() * np.sqrt(365)
        )
    return df


def add_volume_features(df):
    """ボリューム特徴量"""
    # 前日比変化率
    df['vol_chg'] = df.groupby('symbol')['volume'].transform(lambda x: x.pct_change())
    # 7日移動平均比
    sma7 = df.groupby('symbol')['volume'].transform(
        lambda x: x.rolling(7, min_periods=2).mean()
    )
    df['vol_sma_ratio_7d'] = df['volume'] / sma7.replace(0, np.nan)
    # 14日 z スコア
    vol_mean14 = df.groupby('symbol')['volume'].transform(
        lambda x: x.rolling(14, min_periods=5).mean()
    )
    vol_std14  = df.groupby('symbol')['volume'].transform(
        lambda x: x.rolling(14, min_periods=5).std()
    )
    df['volume_zscore_14d'] = (df['volume'] - vol_mean14) / vol_std14.replace(0, np.nan)
    return df


def add_dow_dummies(df):
    """曜日ダミー (暗号資産週末効果)"""
    dow = df['date'].dt.dayofweek  # 0=月曜 ... 6=日曜
    for d in range(7):
        df[f'dow_{d}'] = (dow == d).astype(int)
    return df


def add_btc_leader_effect(df):
    """
    BTC先行効果: BTC-USDのリターンを他の銘柄の特徴量として結合。
    BTC-USD 自身の行には 0 を入れる（自己参照を避ける）。
    """
    btc = df[df['symbol'] == 'BTC-USD'][['date', 'logret_1d', 'logret_3d']].rename(
        columns={'logret_1d': 'BTC_ret_1d', 'logret_3d': 'BTC_ret_3d'}
    )
    df = df.merge(btc, on='date', how='left')
    # BTC-USD 自身の行は 0 にする
    is_btc = df['symbol'] == 'BTC-USD'
    df.loc[is_btc, 'BTC_ret_1d'] = 0.0
    df.loc[is_btc, 'BTC_ret_3d'] = 0.0
    return df


def add_cs_rank(df, cols):
    """クロスセクションランク (日次パーセンタイル)"""
    for col in cols:
        if col in df.columns:
            df[f'{col}_rank'] = df.groupby('date')[col].rank(pct=True)
    return df


def merge_macro(df, df_m):
    """
    マクロ指標を結合し、1日リターンを特徴量として追加。
    暗号資産関連: USD index, gold, USDJPY を主に使用。
    """
    df = df.merge(df_m, on='date', how='left')
    macro_cols = [c for c in df_m.columns if c != 'date']
    df[macro_cols] = df[macro_cols].ffill()
    # 1日リターンのみ追加（暗号資産は短期反応が主）
    for col in macro_cols:
        df[f'{col}_ret_1d'] = df.groupby('symbol')[col].transform(
            lambda x: x.pct_change(1)
        )
    return df


print('関数定義 OK')

In [ ]:
print('特徴量構築中...')
df = df_raw.copy().sort_values(['symbol', 'date']).reset_index(drop=True)

df = add_returns(df, windows=(1, 3, 7, 14, 30))
df = add_forward_returns(df, windows=(1, 7))
df = add_sma(df, windows=(7, 14, 30, 60))
df = add_rsi(df, window=14)
df = add_macd(df)
df = add_bollinger(df)
df = add_volatility(df, windows=(7, 14, 30))
df = add_volume_features(df)
df = add_btc_leader_effect(df)
df = add_dow_dummies(df)
df = add_cs_rank(df, ['ret_1d', 'ret_7d', 'vol_7d', 'rsi_14', 'vol_sma_ratio_7d'])
df = merge_macro(df, df_macro)

print(f'特徴量構築完了: {df.shape[0]:,} 行 x {df.shape[1]} 列')
print(f'銘柄: {sorted(df["symbol"].unique().tolist())}')
df.head(3)

---
## 3. データリーケージチェック

フォワードリターン (`fwd_ret_*`) が特徴量に含まれていないことを確認する。

In [ ]:
# 特徴量リスト定義
ID_COLS = {'symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'asset_class'}
FWD_COLS = {c for c in df.columns if c.startswith('fwd_ret_')}
# マクロ生値列 (リターン派生物は特徴量に含めるが生値は除外)
MACRO_RAW = {c for c in df.columns
             if c in ['nikkei225','dow_jones','usd_index','crude_oil','gold','us10y_yield','usdjpy']}

EXCLUDE = ID_COLS | FWD_COLS | MACRO_RAW
FEATURES = [c for c in df.columns if c not in EXCLUDE]

print('=== データリーケージチェック ===')
leakage_found = []
for fwd in FWD_COLS:
    if fwd in FEATURES:
        leakage_found.append(fwd)

if leakage_found:
    print(f'[ERROR] リーケージ検出: {leakage_found}')
else:
    print('[OK] fwd_ret_* は特徴量に含まれていません')

# フォワードリターン列の確認
for fwd in sorted(FWD_COLS):
    assert fwd not in FEATURES, f'リーケージ: {fwd} が FEATURES に含まれています!'

print(f'\n特徴量数: {len(FEATURES)}')
print(f'ターゲット: {TARGET}')
print(f'\n特徴量カテゴリ確認:')
for prefix in ['ret_', 'logret_', 'sma_', 'ema_', 'price_sma', 'rsi_', 'macd',
               'bb_', 'vol_', 'volume_', 'BTC_', 'dow_', '_rank', '_ret_1d']:
    cols = [c for c in FEATURES if prefix in c]
    if cols:
        print(f'  {prefix:20s}: {len(cols):3d}個  例={cols[:3]}')

---
## 4. ウォークフォワード学習

In [ ]:
def rank_ic(y_true, y_pred):
    """Rank IC (スピアマン相関)"""
    corr, _ = spearmanr(y_true, y_pred)
    return float(corr)


def walkforward_train(df_all, features, target, n_splits=5, params=None):
    """
    ウォークフォワード学習・評価。

    Parameters
    ----------
    df_all   : 全データ (symbol, date, features, target を含む DataFrame)
    features : 特徴量列名リスト
    target   : ターゲット列名
    n_splits : TimeSeriesSplit の分割数
    params   : LGBMRegressor パラメータ dict

    Returns
    -------
    models     : 各 Fold のモデルリスト
    metrics_df : Fold ごとの評価指標 DataFrame
    oof_df     : OOF 予測値 DataFrame
    """
    if params is None:
        params = LGB_PARAMS.copy()

    df_all = df_all.dropna(subset=features + [target]).copy()
    df_all = df_all.sort_values('date').reset_index(drop=True)
    dates  = df_all['date'].sort_values().unique()
    tscv   = TimeSeriesSplit(n_splits=n_splits)

    models, metrics_rows, oof_records = [], [], []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(dates)):
        tr_dates = dates[tr_idx]
        va_dates = dates[va_idx]
        tr_mask  = df_all['date'].isin(tr_dates)
        va_mask  = df_all['date'].isin(va_dates)

        X_tr = df_all.loc[tr_mask, features]
        y_tr = df_all.loc[tr_mask, target]
        X_va = df_all.loc[va_mask, features]
        y_va = df_all.loc[va_mask, target]

        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )

        y_pred = m.predict(X_va)
        ic     = rank_ic(y_va.values, y_pred)
        models.append(m)

        metrics_rows.append({
            'fold':      fold + 1,
            'val_start': pd.Timestamp(va_dates.min()),
            'val_end':   pd.Timestamp(va_dates.max()),
            'rank_ic':   ic,
            'n_train':   int(tr_mask.sum()),
            'n_val':     int(va_mask.sum()),
        })

        oof = df_all.loc[va_mask, ['date', 'symbol', target]].copy()
        oof['pred'] = y_pred
        oof_records.append(oof)

        print(
            f'  Fold {fold+1}: RankIC={ic:+.4f}  '
            f'Val {pd.Timestamp(va_dates.min()).date()} ~ {pd.Timestamp(va_dates.max()).date()}'
        )

    metrics_df = pd.DataFrame(metrics_rows)
    oof_df     = pd.concat(oof_records, ignore_index=True)
    mean_ic    = metrics_df['rank_ic'].mean()
    std_ic     = metrics_df['rank_ic'].std()
    print(f'\n  平均 RankIC: {mean_ic:+.4f} ± {std_ic:.4f}  (目標 >= 0.03)')
    return models, metrics_df, oof_df


print('関数定義 OK')

In [ ]:
# 学習用データ準備
df_model = df.dropna(subset=FEATURES + [TARGET]).copy()

print(f'学習データ: {len(df_model):,} 行  期間: {df_model["date"].min().date()} ~ {df_model["date"].max().date()}')
print(f'銘柄内訳:')
print(df_model.groupby('symbol').size().to_string())
print()
print('=== ウォークフォワード学習開始 (target=fwd_ret_7d) ===')
models, metrics_df, oof_df = walkforward_train(
    df_model, FEATURES, TARGET, n_splits=N_SPLITS
)

---
## 5. 結果評価

In [ ]:
# ---- Fold 別 Rank IC バーチャート ----
fig, ax = plt.subplots(figsize=(10, 4))
colors  = ['#2ca02c' if v > 0 else '#d62728' for v in metrics_df['rank_ic']]
ax.bar(metrics_df['fold'], metrics_df['rank_ic'], color=colors, edgecolor='white', alpha=0.85)

mean_ic = metrics_df['rank_ic'].mean()
ax.axhline(0,       color='black',  linewidth=1)
ax.axhline(0.03,    color='green',  linewidth=1.5, linestyle='--', label='目標 IC=0.03')
ax.axhline(mean_ic, color='orange', linewidth=2,   linestyle='-',
           label=f'平均 IC={mean_ic:+.4f}')

ax.set_xlabel('Fold')
ax.set_ylabel('Rank IC (Spearman)')
ax.set_title('暗号資産 ウォークフォワード Rank IC (target=fwd_ret_7d)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '43_rank_ic_by_fold.png', dpi=120, bbox_inches='tight')
plt.show()

print(metrics_df[['fold', 'val_start', 'val_end', 'rank_ic', 'n_train', 'n_val']].to_string(index=False))

In [ ]:
# ---- 月次 Rank IC ヒートマップ風バーチャート ----
oof_df['ym'] = oof_df['date'].dt.to_period('M')
monthly_ic   = oof_df.groupby('ym').apply(
    lambda g: rank_ic(g[TARGET].values, g['pred'].values)
).rename('rank_ic').reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in monthly_ic['rank_ic']]
ax.bar(range(len(monthly_ic)), monthly_ic['rank_ic'], color=colors, alpha=0.8)
ax.axhline(0, color='black', linewidth=1)
ax.axhline(
    monthly_ic['rank_ic'].mean(), color='orange', linewidth=1.5, linestyle='--',
    label=f'平均={monthly_ic["rank_ic"].mean():+.4f}'
)
step = max(1, len(monthly_ic) // 12)
ax.set_xticks(range(0, len(monthly_ic), step))
ax.set_xticklabels(
    [str(monthly_ic['ym'].iloc[i]) for i in range(0, len(monthly_ic), step)],
    rotation=45, ha='right', fontsize=8
)
ax.set_title('月次 Rank IC (OOF) - 暗号資産')
ax.set_ylabel('Rank IC')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '43_monthly_rank_ic.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'月次 IC 正率: {(monthly_ic["rank_ic"] > 0).mean():.1%}')

In [ ]:
# ---- 特徴量重要度 Top 25 ----
imp_list = [
    pd.Series(m.feature_importances_, index=FEATURES, name=f'fold{i+1}')
    for i, m in enumerate(models)
]
imp_df = pd.concat(imp_list, axis=1)
imp_df['mean_importance'] = imp_df.mean(axis=1)
imp_df = imp_df.sort_values('mean_importance', ascending=False).reset_index()
imp_df.columns = ['feature'] + [f'fold{i+1}' for i in range(len(models))] + ['mean_importance']

TOP_N   = 25
top_imp = imp_df.head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top_imp['feature'][::-1], top_imp['mean_importance'][::-1],
        color='steelblue', alpha=0.85)
ax.set_title(f'暗号資産 特徴量重要度 Top {TOP_N} (Fold 平均)')
ax.set_xlabel('Importance')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '43_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print(top_imp[['feature', 'mean_importance']].head(20).to_string(index=False))

In [ ]:
# ---- 株式 vs 暗号資産 IC 比較テーブル ----
# 株式の結果が保存されている場合は読み込む
equity_metrics_path = TABLES_DIR / 'walkforward_metrics.csv'

crypto_summary = {
    'asset':   'Crypto (BTC/ETH/SOL/BNB/XRP)',
    'target':  TARGET,
    'mean_ic': metrics_df['rank_ic'].mean(),
    'std_ic':  metrics_df['rank_ic'].std(),
    'min_ic':  metrics_df['rank_ic'].min(),
    'max_ic':  metrics_df['rank_ic'].max(),
    'positive_folds': int((metrics_df['rank_ic'] > 0).sum()),
    'target_met': metrics_df['rank_ic'].mean() >= 0.03,
}

rows = [crypto_summary]

if equity_metrics_path.exists():
    eq_metrics = pd.read_csv(equity_metrics_path)
    equity_summary = {
        'asset':   'Equity (JP stocks)',
        'target':  'fwd_ret_5d',
        'mean_ic': eq_metrics['rank_ic'].mean(),
        'std_ic':  eq_metrics['rank_ic'].std(),
        'min_ic':  eq_metrics['rank_ic'].min(),
        'max_ic':  eq_metrics['rank_ic'].max(),
        'positive_folds': int((eq_metrics['rank_ic'] > 0).sum()),
        'target_met': eq_metrics['rank_ic'].mean() >= 0.03,
    }
    rows.append(equity_summary)

compare_df = pd.DataFrame(rows)
print('=== 株式 vs 暗号資産 Rank IC 比較 (目標 >= 0.03) ===')
print(compare_df.to_string(index=False))

---
## 6. 最新日の予測 (Inference)

In [ ]:
latest_date = df['date'].max()
df_latest   = df[df['date'] == latest_date].copy()

avail_feats = [f for f in FEATURES if f in df_latest.columns]
df_latest   = df_latest.dropna(subset=avail_feats)

if df_latest.empty:
    print('最新日のデータが不足しています')
else:
    X_latest   = df_latest[FEATURES].copy()
    # 全 Fold モデルのアンサンブル予測
    preds_all  = np.array([m.predict(X_latest) for m in models])
    preds_mean = preds_all.mean(axis=0)

    df_pred = df_latest[['symbol']].copy()
    df_pred['predicted_logret_7d'] = preds_mean
    df_pred['predicted_ret_7d_pct'] = (np.exp(preds_mean) - 1) * 100  # パーセント換算
    df_pred = df_pred.sort_values('predicted_logret_7d', ascending=False).reset_index(drop=True)
    df_pred.insert(0, 'rank', range(1, len(df_pred) + 1))

    print(f'予測基準日: {latest_date.date()}')
    print(f'予測銘柄数: {len(df_pred)}')
    print()
    print('=== 全銘柄ランキング (7日先予測リターン) ===')
    print(df_pred[['rank', 'symbol', 'predicted_logret_7d', 'predicted_ret_7d_pct']].to_string(index=False))

    # 保存
    pred_path = TABLES_DIR / 'crypto_latest_predictions.csv'
    df_pred.to_csv(pred_path, index=False, encoding='utf-8-sig')
    print(f'\n予測保存: {pred_path}')

In [ ]:
if not df_latest.empty and len(df_pred) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#2ca02c' if v > 0 else '#d62728' for v in df_pred['predicted_logret_7d']]
    ax.barh(
        df_pred['symbol'][::-1],
        df_pred['predicted_ret_7d_pct'][::-1],
        color=colors[::-1], alpha=0.85
    )
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('予測 7日先リターン (%)')
    ax.set_title(f'暗号資産 予測ランキング  (as of {latest_date.date()})')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '43_latest_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## 7. アーティファクト保存

In [ ]:
# モデル保存
model_path = TABLES_DIR / 'crypto_lgbm_models.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({'models': models, 'feature_cols': FEATURES, 'target': TARGET}, f)
print(f'モデル保存        : {model_path}')

# 特徴量リスト保存
feat_path = TABLES_DIR / 'crypto_features.csv'
pd.DataFrame({'feature': FEATURES}).to_csv(feat_path, index=False, encoding='utf-8-sig')
print(f'特徴量リスト保存  : {feat_path}')

# OOF 予測値保存
oof_path = TABLES_DIR / 'crypto_oof_predictions.csv'
oof_df.to_csv(oof_path, index=False, encoding='utf-8-sig')
print(f'OOF 予測値保存    : {oof_path}')

# 評価指標保存
metrics_path = TABLES_DIR / 'crypto_walkforward_metrics.csv'
metrics_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print(f'評価指標保存      : {metrics_path}')

print()
print(f'=== 完了 ===' )
print(f'Folds: {len(models)}  平均 RankIC: {metrics_df["rank_ic"].mean():+.4f}')
print(f'目標 (>= 0.03): {"達成" if metrics_df["rank_ic"].mean() >= 0.03 else "未達成"}')